In [ ]:
import os
import json
from wsgiref import headers

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from bs4 import BeautifulSoup

print("Loading environment variables...")
load_dotenv(override=True)

requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_links(url):
    """
    Returns the links on the website on the given url
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


links = fetch_website_links("https://edwarddonner.com")
links



Loading environment variables...


['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [71]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://example.com/about"},
        {"type": "careers page", "url": "https://example.com/careers"}
    ]
}
"""

In [96]:
def get_links_user_prompt(url):
    user_prompt = f"""
You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

"""
    
    links = fetch_website_links(url)
    user_prompt += f"Here are the links on the website: {links}"
    print(user_prompt)
    return user_prompt

In [97]:
print(get_links_user_prompt("https://edwarddonner.com"))


You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

Here are the links on the website: ['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/2026/02/17/ai-cod

In [74]:
def select_relevant_links(url):
    user_prompt = get_links_user_prompt(url)
    response = ollama.chat.completions.create(model="gemma3:270m", 
    messages=[
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": user_prompt}
    ], 
    response_format={"type": "json_object"})
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [75]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/',
   'title': 'About Me and About Nebula',
   'content': 'About me and about nebula - A short introduction to our company.'},
  {'type': 'company page',
   'url': 'https://edwarddonner.com/careers',
   'title': 'Our Careers / Jobs Page',
   'description': 'We have a team of passionate professionals dedicated to helping you achieve your career goals. Explore our range of exciting opportunities and find the perfect fit for you.'},
  {'type': 'career page',
   'url': 'https://edwarddonner.com/careers',
   'title': 'Our Career / Jobs Page',
   'description': 'We have a dedicated team of professionals guiding you through the job search process, from application to interview. Discover how we can help you land your dream role.'},
  {'type': 'program page',
   'url': 'https://edwarddonner.com/program',
   'title': 'Our Program / Careers Page',
   'description': 'We offer a wide range of programs to

In [76]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [77]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    print("Relevant Links:")
    print(relevant_links)
    print(type(relevant_links))
    print(relevant_links.keys())

    result = f"## Landing Page Content\n\n{contents}\n\n## Relevant Links\n\n"

    for link in relevant_links["links"]:
        result += f"\n\n## Link : {link['type']}\n"
        result += fetch_website_contents(link["url"])

    return result

In [98]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))


You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

Here are the links on the website: ['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/2026/02/17/ai-cod

In [89]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [91]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [92]:
get_brochure_user_prompt("Edward Donner", "https://edwarddonner.com")

Relevant Links:
{'links': [{'type': 'about page', 'url': 'https://example.com/about'}, {'type': 'careers page', 'url': 'https://example.com/careers'}]}
<class 'dict'>
dict_keys(['links'])


'\nYou are looking at a company called: Edward Donner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page Content\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu 

In [94]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[      
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [95]:
create_brochure("Edward Donner", "https://edwarddonner.com")

Relevant Links:
{'links': [{'type': 'about page', 'url': 'https://example.com/about'}, {'type': 'careers page', 'url': 'https://example.com/careers'}]}
<class 'dict'>
dict_keys(['links'])


# Edward Donner  
## Pioneering the Future of Large‑Language Models  

---

### Who We Are  

Edward Donner is an AI thought‑leader, coder, and entrepreneur who transforms how people build and deploy LLMs. With a background that spans **JPMorgan**, the buy‑out of his first startup *untapt* in 2021, and now the co‑founding of **Nebula.io**, he brings industry scale expertise to an open‑source, community‑driven framework.  

---

### Core Offerings  

| Area | What We Deliver | Why It Matters |
|------|-----------------|----------------|
| **LLM Arena – “Outsmart”** | A public platform where LLMs compete in diplomacy, strategy and humor. | Fast, hands‑on benchmarking for research, product demos & educational purposes. |
| **Nebula.io** | Production‑grade micro‑services that wrap LLMs in an agentic architecture. | Turn AI ideas into scalable apps without the overhead of monolithic frameworks. |
| **Educational Curriculum** | 200+ self‑paced courses on Udemy covering everything from “AI Coder” to “Voice Agent Builders.” | Over 600 000 students enrolled, spanning 194 countries – a validated learning product that fuels our talent pipeline. |

---

### Company Culture  

- **Experimentation at the Core:** Code, tweak, iterate—then share results back with the community (e.g., the “Outsmart” battles).  
- **Learning & Sharing:** We host public lectures, write tutorials, and keep a digital avatar on the site to answer questions live.  
- **Community‑First Mindset:** Courses are open‑source where possible; ideas are challenged openly in social media threads and Hacker News discussions.  
- **Tech + Creativity:** Founder enjoys amateur electronic music production – we encourage multi‑disciplinary exploration among all team members.

---

### Why Choose Edward Donner?

| Audience | Value Proposition |
|----------|-------------------|
| **Students & Hobbyists** | Free or low‑cost courses that give industry‑relevant skills in AI and LLM engineering. |
| **Businesses needing AI solutions** | Nebula.io accelerates the time to production for custom agents, voice assistants, and data insights. |
| **Investors** | Proven track record of scaling a startup (untapt acquisition), deep expertise in high‑growth AI domains, and a large global learning community that creates demand for our products. |
| **Talent & Recruiters** | A tech environment where you can work on LLMs from research to production, collaborate with peers worldwide, and influence next‑generation AI tools. |

---

### Careers

While we don’t list specific roles in the given pages, we are continuously looking for:

- **Software Engineers (Python / Rust)** – Building high‑performance LLM services.  
- **ML Research Scientists** – Developing novel model architectures and battle strategies.  
- **Product & Community Managers** – Growing our educational ecosystem and customer base.

> *Think you could help shape the next wave of AI? Join us and code the future.*

---

### Investor Highlights  

- **Untapt Acquired (2021)** – Validated exit, proven market fit.  
- **600 k+ Udemy Enrollments** – Demonstrates worldwide demand for our IP.  
- **Global Presence** – Courses taught in 194 countries; Nebula.io serving enterprise deployments across the globe.  

---

### Get In Touch

| Medium | Contact |
|--------|---------|
| Email | ed@edwarddonner.com |
| Website | [www.edwarddonner.com](https://www.edwarddonner.com) |
| LinkedIn | [Edward Donner](#) |
| Twitter | @EdwardDonner |
| Facebook | /EdwardDonner |

---

> **“Let’s create the future of LLMs together.”**  
&nbsp; – Edward Donner

---